## CHULETA ANOVA | Probabilidad y Estadística iMAT-ICAI | versión 2.1

## Librerías necesarias (requiere anova.py junto al notebook)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from more_itertools import locate

from anova import anova1, anova2

In [ ]:
# asegura que pandas mostrará todo el dataframe completo (100 filas)
# necesario para visualizar bien modelo de medias de ANOVA
pd.set_option('display.max_rows', 100)

## Dataframe de ejemplo

In [ ]:
# datos de ejemplo (dataframe df)

rng = np.random.default_rng(999)
y = [1, 2, 2, 1, 2, 3, 2, 3, 1, 1, 3, 3, 3, 3, 2, 2, 2, 3, 1, 1, 1, 2, 3, 3, 2, 1, 2, 3, 2, 1]
z = ['a', 'a', 'b', 'd', 'd', 'd', 'd','a', 'b', 'b','a', 'a', 'b', 'c', 'b', 'a', 'b','c', 'd', 'b','c','d','a', 'c', 'a', 'b', 'a', 'd', 'c', 'b']
x =  y + 0.5 * rng.random(30) # salida, creada a partir del factor y

df =  pd.DataFrame(list(zip(x, y, z)), columns=['VAR1', 'VAR2', 'VAR3'])

# muestra df
df

## 1. ANOVA de un factor

In [ ]:
# se establece la fórmula indicando los nombres de la salida y el factor de entrada
# formula = 'SALIDA ~ C(FACTOR)'
# en la fórmula C() indica que la variable se trate como una variable categórica

salida = 'VAR1'
factor = 'VAR3'
formula = f'{salida} ~ C({factor})'
print('Fórmula: ', formula)

# calcula ANOVA
tabla_anova, modelo_medias = anova1(formula, df)

# muestra tabla anova
print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

In [ ]:
# muestra modelo de medias
print('Modelo de medias:')
print(modelo_medias)

# gráficamente
fig = plt.figure(1, figsize=(15, 4))
plt.subplot(1,2,1)
plt.bar(modelo_medias.index[0], modelo_medias.Mean.iloc[0])
plt.ylabel('Media global')

plt.subplot(1,2,2)
plt.bar(modelo_medias.index[1:], modelo_medias.Mean.iloc[1:])
plt.ylabel('Coeficientes alfa')
plt.xlabel(factor)
plt.show()

## 2. ANOVA de dos factores

### 2.1 Sin interacción (usando +)

In [ ]:
# se establece la fórmula indicando los nombres de la salida y los dos factores de entrada
# formula = 'SALIDA ~ C(FACTOR_1) + C(FACTOR_2)'
# en la fórmula C() indica que la variable se trate como una variable categórica

salida = 'VAR1'
factor_1 = 'VAR2'
factor_2 = 'VAR3'
formula = f'{salida} ~ C({factor_1}) + C({factor_2})'
print('Fórmula: ', formula)

# calcula ANOVA 2 (sin interacción)
tabla_anova, modelo_medias = anova2(formula, df)

# muestra tabla anova
print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

# muestra modelo de medias
print('Modelo de medias:')
print(modelo_medias)

#### 2.1.1 Gráficas modelo de medias

In [ ]:
# gráficamente modelo de medias
ind_factor_1 = np.where(modelo_medias.index.str.startswith(factor_1))[0].tolist()
ind_factor_2 = np.where(modelo_medias.index.str.startswith(factor_2))[0].tolist()

gr_factor_1 = modelo_medias.index[ind_factor_1].str.replace('.*_', '', regex =True)
gr_factor_2 = modelo_medias.index[ind_factor_2].str.replace('.*_', '', regex =True)


fig = plt.figure(1, figsize=(20, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias.index[0], modelo_medias.Mean.iloc[0])
plt.ylabel('Media global')

plt.subplot(1,3,2)
plt.bar(gr_factor_1, modelo_medias.Mean.iloc[ind_factor_1])
plt.ylabel('Coeficientes alfa')
plt.xlabel(factor_1)

plt.subplot(1,3,3)
plt.bar(gr_factor_2, modelo_medias.Mean.iloc[ind_factor_2])
plt.ylabel('Coeficientes beta')
plt.xlabel(factor_2)
plt.show()

In [ ]:
# gráfica 3D
X, Y = np.mgrid[0:len(ind_factor_1), 0:len(ind_factor_2)]

# evalua el modelo de medias en cada punto (x_i,y_j)
Z = np.zeros(X.shape)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
         Z[i,j] =  modelo_medias.Mean.iloc[0] + modelo_medias.Mean.iloc[ind_factor_1].iloc[i] +  modelo_medias.Mean.iloc[ind_factor_2].iloc[j]
 

fig = plt.figure(2, figsize=(8, 8))
ax1 = plt.axes(projection='3d')
ax1.set_xticks(range(len(ind_factor_1)))
ax1.set_yticks(range(len(ind_factor_2)))

ax1.set_xticklabels(gr_factor_1)
ax1.set_yticklabels(gr_factor_2)
ax1.plot_wireframe(X, Y, Z, color='red')
ax1.stem(X.flatten(), Y.flatten(), Z.flatten(), basefmt='none')
plt.xlabel(factor_1); plt.ylabel(factor_2);
plt.title(f'Modelo de Medias ANOVA:  {formula}')
plt.show()

### 2.2 Con interacción

In [ ]:
# se establece la fórmula indicando los nombres de la salida y los dos factores de entrada
# formula = 'SALIDA ~ C(FACTOR_1) * C(FACTOR_2)'
# en la fórmula C() indica que la variable se trate como una variable categórica
# el * significa que lleva INTERACCIÓN entre ambos factores

salida = 'VAR1'
factor_1 = 'VAR2'
factor_2 = 'VAR3'
formula = f'{salida} ~ C({factor_1}) * C({factor_2})'
print('Fórmula: ', formula)

# calcula ANOVA 2 (CON interacción)
tabla_anova, modelo_medias = anova2(formula, df) 

# muestra tabla anova
print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

# muestra modelo de medias
print('Modelo de medias:')
print(modelo_medias)

#### 2.2.1 Gráficas modelo de medias

In [ ]:
# Calcular coeficientes interacción

modelo_medias_not_inter = (
    modelo_medias
    .loc[lambda a: ~a.index.str.contains(":")]
)

modelo_medias_inter = (
    modelo_medias
    .loc[lambda a: a.index.str.contains(":")]
    .assign(
        nombre_f1 = lambda a: a.index.str.split(" : ").str[0],
        nombre_f2 = lambda a: a.index.str.split(" : ").str[1]
    )
)

ind_factor_1 = np.where(modelo_medias_not_inter.index.str.startswith(factor_1))[0].tolist()
ind_factor_2 = np.where(modelo_medias_not_inter.index.str.startswith(factor_2))[0].tolist()

gr_factor_1 = modelo_medias_not_inter.index[ind_factor_1].str.replace('.*_', '', regex =True)
gr_factor_2 = modelo_medias_not_inter.index[ind_factor_2].str.replace('.*_', '', regex =True)


fig = plt.figure(1, figsize=(20, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias_not_inter.index[0], modelo_medias_not_inter.Mean.iloc[0])
plt.ylabel('Media global')

plt.subplot(1,3,2)
plt.bar(gr_factor_1, modelo_medias_not_inter.Mean.iloc[ind_factor_1])
plt.ylabel('Coeficientes alfa')
plt.xlabel(factor_1)

plt.subplot(1,3,3)
plt.bar(gr_factor_2, modelo_medias_not_inter.Mean.iloc[ind_factor_2])
plt.ylabel('Coeficientes beta')
plt.xlabel(factor_2)
plt.show()

fig = plt.figure(2, figsize=(40, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias_inter.index, modelo_medias_inter.Mean)
plt.ylabel('Coeficientes theta')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# gráfica 3D
X, Y = np.mgrid[0:len(ind_factor_1), 0:len(ind_factor_2)]

# evalua el modelo de medias en cada punto (x_i,y_j)
Z = np.zeros(X.shape)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        factor_1_name = modelo_medias_not_inter.iloc[ind_factor_1].index[i]
        factor_2_name = modelo_medias_not_inter.iloc[ind_factor_2].index[j]
        interaction = modelo_medias_inter.loc[lambda a: (a['nombre_f1'] == factor_1_name) & (a['nombre_f2'] == factor_2_name)].Mean.iloc[0]
        Z[i,j] =  modelo_medias_not_inter.Mean.iloc[0] + modelo_medias_not_inter.Mean.iloc[ind_factor_1].iloc[i] +  modelo_medias_not_inter.Mean.iloc[ind_factor_2].iloc[j] + interaction

fig = plt.figure(2, figsize=(8, 8))
  
ax1 = plt.axes(projection='3d')
ax1.set_xticks(range(len(ind_factor_1)))
ax1.set_yticks(range(len(ind_factor_2)))

ax1.set_xticklabels(gr_factor_1)
ax1.set_yticklabels(gr_factor_2)
ax1.plot_wireframe(X, Y, Z, color='green')
ax1.stem(X.flatten(), Y.flatten(), Z.flatten(), basefmt='none')
plt.xlabel(factor_1); plt.ylabel(factor_2);
plt.title(f'Modelo de Medias ANOVA:  {formula}')
plt.show()